In [15]:
from pydantic import BaseModel,Field
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode
from langchain_core.messages import AIMessage,ToolMessage
from langchain_core.messages import convert_to_messages ,convert_to_openai_messages

from jinja2 import  Template
from typing import Literal,Dict,Any,Annotated,List
from IPython.display import Image,display
from operator import add
from openai import  OpenAI
import numpy as np

import random
import ast
import inspect
import instructor
import json
from langchain_core.messages import AIMessage,ToolMessage,convert_to_openai_messages,HumanMessage,SystemMessage
from qdrant_client import QdrantClient
from qdrant_client.models import Distance,VectorParams,PointStruct,Prefetch,FieldCondition,MatchText,FusionQuery,Document,Filter,MatchValue
import openai
import os
from langchain_openai import ChatOpenAI
from utils.utils import format_ai_message,get_tool_descriptions,get_type_from_annotation
import psycopg2

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

### ADD To SHOPPING CART TOOL

In [11]:
items = [
    {
        "product_id": "B0BH41HYFZ",
        "quantity": 2
    },
    {
        "product_id": "B0BKPB2YQ9",
        "quantity": 4
    }
]

In [14]:
from psycopg2.extras import RealDictCursor

def add_to_shopping_cart(items: list[dict], user_id: str, cart_id: str) -> str:

    """Add a list of provided items to the shopping cart.
    
    Args:
        items: A list of items to add to the shopping cart. Each item is a dictionary with the following keys: product_id, quantity.
        user_id: The id of the user to add the items to the shopping cart.
        cart_id: The id of the shopping cart to add the items to.
        
    Returns:
        A list of the items added to the shopping cart.
    """

    conn = psycopg2.connect(
        host=os.getenv("DB_HOST"),
        port=os.getenv("DB_PORT"),
        database="tool_database",
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD")
    )
    conn.autocommit = True

    with conn.cursor(cursor_factory=RealDictCursor) as cursor:
        
        for item in items:
            product_id = item['product_id']
            quantity = item['quantity']

            qdrant_client = QdrantClient(url="http://localhost:6333")

            dummy_vector = np.zeros(1536).tolist()
            payload = qdrant_client.query_points(
                collection_name="Amazon-items-collection-02-hybrid-serach",
                prefetch=[
                    Prefetch(
                        query=dummy_vector,
                        filter=Filter(
                            must=[
                                FieldCondition(
                                    key="parent_asin",
                                    match=MatchValue(value=product_id)
                                )
                            ]
                    ),
                        using="text-embedding-model-3-small",
                        limit=20
                    )
                ],
                query=FusionQuery(fusion="rrf"),
                limit=1,
            )
            if not payload.points:
                print(f"Product {product_id} not found in Qdrant; skipping.")
                continue
            payload = payload.points[0].payload

            product_image_url = payload.get("image")
            price = payload.get("price")
            currency = 'USD'
        
            # Check if item already exists
            check_query = """
                SELECT id, quantity, price 
                FROM shopping_carts.shopping_cart_items 
                WHERE user_id = %s AND shopping_cart_id = %s AND product_id = %s
            """
            cursor.execute(check_query, (user_id, cart_id, product_id))
            existing_item = cursor.fetchone()
            
            if existing_item:
                # Update existing item
                new_quantity = existing_item['quantity'] + quantity
                
                update_query = """
                    UPDATE shopping_carts.shopping_cart_items 
                    SET 
                        quantity = %s,
                        price = %s,
                        currency = %s,
                        product_image_url = COALESCE(%s, product_image_url)
                    WHERE user_id = %s AND shopping_cart_id = %s AND product_id = %s
                    RETURNING id, quantity, price
                """
                
                cursor.execute(update_query, (new_quantity, price, currency, product_image_url, user_id, cart_id, product_id))
            
            else:
                # Insert new item
                insert_query = """
                    INSERT INTO shopping_carts.shopping_cart_items (
                        user_id, shopping_cart_id, product_id,
                        price, quantity, currency, product_image_url
                    ) VALUES (%s, %s, %s, %s, %s, %s, %s)
                    RETURNING id, quantity, price
                """
                
                cursor.execute(insert_query, (user_id, cart_id, product_id, price, quantity, currency, product_image_url))
            
    return f"Added {items} to the shopping cart."


In [ ]:
add_to_shopping_cart(items=items,user_id="abc",cart_id="def")

### Get the Shoppping Cart items Tool

In [21]:
def get_shopping_cart(user_id:str,cart_id:str)->list[dict]:
    """
    Reterive all items in a user's shopping cart
     
    Args:
      user_id:User ID
      cart_id:Cart identifier

    Return:
      List of dictionaries conatining cart items
    """
    conn = psycopg2.connect(
        host=os.getenv("DB_HOST"),
        port=os.getenv("DB_PORT"),
        database="tool_database",
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD")
    )
    conn.autocommit = True

    with conn.cursor(cursor_factory=RealDictCursor) as cursor:
      query="""
        SELECT 
           product_id,price,quantity,
           currency,product_image_url,
           (price * quantity) as total_price
          FROM shopping_carts.shopping_cart_items
          WHERE user_id=%s AND shopping_cart_id=%s
          ORDER BY added_at DESC
      """
      cursor.execute(query,(user_id,cart_id))
      return [dict(row) for row in cursor.fetchall()]



In [22]:
get_shopping_cart(user_id="abc",cart_id="def")

[{'product_id': 'B0BKPB2YQ9',
  'price': Decimal('14.99'),
  'quantity': 8,
  'currency': 'USD',
  'product_image_url': 'https://m.media-amazon.com/images/I/41WsGRr-3TL._AC_.jpg',
  'total_price': Decimal('119.92')}]

 Availibility Deleting from shopping cart tool

In [23]:
def remove_from_cart(product_id:str,user_id:str,cart_id:str)->str:
    """Remove an item Completely from the Shopping Cart,
    Args:
      user_id:user ID
      product_id:Product Id to remove
      cart_id: Cart Identifier

    Return:
      True if item was removed ,False if item wasn't found

    """
    conn = psycopg2.connect(
        host=os.getenv("DB_HOST"),
        port=os.getenv("DB_PORT"),
        database="tool_database",
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD")
    )
    conn.autocommit = True
    with conn.cursor(cursor_factory=RealDictCursor) as cursor:
      query="""
        DELETE FROM shopping_cart.shopping_cart_items
        WHERE user_id =%s AND shopping_cart_id=%s AND product_id=%s
      """
      cursor.execute(query,(user_id,cart_id,product_id))

    return cursor.rowcount>0